In [1]:
import torch

data = torch.load('wine-red.pt')

In [2]:
import pandas as pd

df = pd.DataFrame(data.numpy())

In [3]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5.0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5.0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5.0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6.0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5.0


In [4]:
X = data[ : , :-1]
y = data[ : , -1: ]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [7]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

C:\Users\googl\AppData\Local\Temp\ipykernel_16444\52480962.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
C:\Users\googl\AppData\Local\Temp\ipykernel_16444\52480962.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(y_val, dtype=torch.float32)


In [8]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [9]:
class WineRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(WineRegressionNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

In [10]:
model = WineRegressionNet(input_dim=11)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [11]:
epochs = 100

print("Старт обучения...")
for epoch in range(epochs):
    model.train()

    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)

        loss.backward()

        optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Эпоха {epoch+1}/{epochs} | Ошибка (MSE Loss): {loss.item():.4f}")

print("\nОбучение завершено!")

Старт обучения...
Эпоха 20/100 | Ошибка (MSE Loss): 0.0009
Эпоха 40/100 | Ошибка (MSE Loss): 0.0647
Эпоха 60/100 | Ошибка (MSE Loss): 0.0028
Эпоха 80/100 | Ошибка (MSE Loss): 0.0000
Эпоха 100/100 | Ошибка (MSE Loss): 1.0370

Обучение завершено!


In [12]:
score = loss**0.5
print(f'Погрешность в {score:.2f} балла')

Погрешность в 1.02 балла
